# FurnForge AI — Sketch-to-Photoreal Furniture Renderer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

> **Recommended Colab Runtime:** Go to **Runtime → Change runtime type → Hardware accelerator → `T4 GPU`** for ~6–10 second photorealistic diffusion rendering. (CPU fallback with 4-step LCM-LoRA distillation is also supported automatically.)

**FurnForge AI** translates hand-drawn furniture sketches into photorealistic studio product renders using five integrated Generative AI & Computer Vision techniques:

1. **Computer Vision (OpenCV):** Bilateral filtering (noise removal preserving crisp strokes), CLAHE adaptive normalization, aspect-ratio preserved 512×512 padding, Canny edge extraction, and morphological closing.
2. **Convolutional Neural Network (ResNet-50 + Grad-CAM):** Pretrained deep CNN visual feature extraction, zero-shot furniture classification priors, lateral symmetry calculation, and pixel-by-pixel **Grad-CAM** attention heatmaps over `layer4`.
3. **Diffusion Models (Stable Diffusion 1.5):** Text-conditioned latent reverse diffusion with CLIP-aligned furniture material prompts.
4. **ControlNet (`lllyasviel/control_v11p_sd15_canny`):** Zero-initialized residual blocks locking exact sketch proportions, leg counts, and structural geometry.
5. **Image-to-Image Translation & LCM-LoRA:** Supports both pure ControlNet structural translation and hybrid ControlNet + Img2Img latent conditioning, plus fast Latent Consistency Model (LCM-LoRA) distillation.


## 1. Environment Check & Dependency Setup

In [ ]:
#@title Check GPU/CPU Hardware & Install Dependencies
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# If running inside a local project checkout with cached weights, reuse them automatically
for env_var, rel_candidates in [
    ("HF_HOME", ["backend/.hf_cache", ".hf_cache"]),
    ("TORCH_HOME", ["backend/.torch_cache", ".torch_cache"]),
]:
    if env_var not in os.environ:
        for cand in rel_candidates:
            if os.path.exists(cand):
                os.environ[env_var] = os.path.abspath(cand)
                break

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

required = {
    "torch": "torch",
    "torchvision": "torchvision",
    "diffusers": "diffusers>=0.28.0",
    "transformers": "transformers>=4.40.0",
    "accelerate": "accelerate>=0.30.0",
    "peft": "peft>=0.11.0",
    "safetensors": "safetensors",
    "cv2": "opencv-python-headless",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
}

missing = []
for mod_name, pkg_spec in required.items():
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pkg_spec)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All core libraries are already installed.")

import torch
import diffusers
import transformers
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Hardware: CUDA GPU ({gpu_name}, {vram_gb:.1f} GB VRAM) | Precision: float16")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = "xpu"
    DTYPE = torch.float16
    print("Hardware: Intel XPU | Precision: float16")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
    print("Hardware: Apple MPS | Precision: float32")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("Hardware: CPU | Precision: float32 (LCM-LoRA 4-step fast mode recommended)")

print(f"PyTorch: {torch.__version__} | Diffusers: {diffusers.__version__} | OpenCV: {cv2.__version__}")
OUTPUT_DIR = Path("furnforge_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Technique 1 — OpenCV Computer Vision Sketch Preprocessor
Cleans pencil paper grain with edge-preserving **bilateral filtering**, enhances line contrast via **CLAHE**, maintains furniture proportions with **aspect-ratio preserved padding (512×512)**, and extracts crisp **Canny edges** with morphological gap closing for ControlNet.

In [ ]:
class OpenCVPreprocessor:
    """OpenCV preprocessing and edge extraction pipeline for furniture sketches."""

    @staticmethod
    def ensure_bgr(img: np.ndarray) -> np.ndarray:
        if img is None:
            raise ValueError("Input image is None.")
        if len(img.shape) == 3 and img.shape[2] == 4:
            alpha = img[:, :, 3:4] / 255.0
            bgr = img[:, :, :3]
            white = np.ones_like(bgr, dtype=np.uint8) * 255
            return (bgr * alpha + white * (1.0 - alpha)).astype(np.uint8)
        if len(img.shape) == 2:
            return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        return img[:, :, :3]

    @staticmethod
    def resize_and_pad(img: np.ndarray, target_size: int = 512, pad_color: int = 255):
        h, w = img.shape[:2]
        scale = target_size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LANCZOS4
        resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

        pad_top = (target_size - new_h) // 2
        pad_bottom = target_size - new_h - pad_top
        pad_left = (target_size - new_w) // 2
        pad_right = target_size - new_w - pad_left

        color_val = [pad_color, pad_color, pad_color] if len(img.shape) == 3 else pad_color
        padded = cv2.copyMakeBorder(
            resized, pad_top, pad_bottom, pad_left, pad_right,
            cv2.BORDER_CONSTANT, value=color_val
        )
        return padded, {"original_size": (w, h), "scaled_size": (new_w, new_h), "scale": round(scale, 4)}

    @classmethod
    def process_sketch(
        cls,
        bgr_img: np.ndarray,
        target_size: int = 512,
        low_threshold: int = 100,
        high_threshold: int = 200,
        blur_kernel: int = 3,
        method: str = "canny",
    ):
        bgr_img = cls.ensure_bgr(bgr_img)
        padded_bgr, pad_meta = cls.resize_and_pad(bgr_img, target_size=target_size, pad_color=255)
        gray = cv2.cvtColor(padded_bgr, cv2.COLOR_BGR2GRAY)
        is_light_bg = float(np.mean(gray)) > 127.0

        # Bilateral filter smooths paper texture while keeping pencil boundaries crisp
        denoised = cv2.bilateralFilter(gray, d=7, sigmaColor=50, sigmaSpace=50)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(denoised)

        if method == "adaptive_threshold":
            flag = cv2.THRESH_BINARY_INV if is_light_bg else cv2.THRESH_BINARY
            edge_map = cv2.adaptiveThreshold(
                enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, flag, 15, 4
            )
        elif method == "lineart":
            inv = (255 - enhanced) if is_light_bg else enhanced
            edge_map = cv2.dilate(inv, np.ones((2, 2), np.uint8), iterations=1)
        else:
            k = blur_kernel if blur_kernel % 2 == 1 else blur_kernel + 1
            blur = cv2.GaussianBlur(enhanced, (k, k), 0)
            edge_map = cv2.Canny(blur, low_threshold, high_threshold)

        morph_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        edge_map = cv2.morphologyEx(edge_map, cv2.MORPH_CLOSE, morph_kernel)

        edge_density = round((int(np.count_nonzero(edge_map)) / (target_size * target_size)) * 100.0, 2)
        contours, _ = cv2.findContours(edge_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        display_sketch = 255 - edge_map

        return {
            "padded_bgr": padded_bgr,
            "edge_map": edge_map,
            "display_sketch": display_sketch,
            "edge_density_percent": edge_density,
            "contour_count": len(contours),
            "pad_metadata": pad_meta,
            "detected_bg": "light (paper)" if is_light_bg else "dark",
        }


def generate_sample_sketch(category: str = "Chair") -> np.ndarray:
    """Generate clean 512x512 furniture line sketches for immediate testing in Colab."""
    canvas = np.ones((512, 512, 3), dtype=np.uint8) * 255
    col = (40, 40, 45)
    t = 3

    if category == "Sofa":
        pts_back = np.array([[90, 200], [256, 175], [422, 200], [435, 280], [77, 280]], np.int32)
        cv2.polylines(canvas, [pts_back], True, col, t, cv2.LINE_AA)
        cv2.ellipse(canvas, (95, 275), (28, 45), 0, 0, 360, col, t, cv2.LINE_AA)
        cv2.ellipse(canvas, (417, 275), (28, 45), 0, 0, 360, col, t, cv2.LINE_AA)
        pts_seat = np.array([[95, 280], [417, 280], [420, 360], [92, 360]], np.int32)
        cv2.polylines(canvas, [pts_seat], True, col, t, cv2.LINE_AA)
        cv2.line(canvas, (202, 200), (202, 360), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (310, 200), (310, 360), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (92, 320), (420, 320), col, 2, cv2.LINE_AA)
        for lx in (115, 256, 397):
            cv2.line(canvas, (lx, 360), (lx, 400), col, 5, cv2.LINE_AA)
    elif category == "Table":
        cv2.ellipse(canvas, (256, 210), (180, 55), 0, 0, 360, col, t, cv2.LINE_AA)
        cv2.ellipse(canvas, (256, 222), (180, 55), 0, 0, 180, col, t, cv2.LINE_AA)
        cv2.line(canvas, (76, 210), (76, 222), col, t, cv2.LINE_AA)
        cv2.line(canvas, (436, 210), (436, 222), col, t, cv2.LINE_AA)
        cv2.line(canvas, (215, 222), (205, 375), col, t, cv2.LINE_AA)
        cv2.line(canvas, (297, 222), (307, 375), col, t, cv2.LINE_AA)
        cv2.line(canvas, (235, 222), (233, 375), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (256, 222), (256, 375), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (277, 222), (279, 375), col, 2, cv2.LINE_AA)
        cv2.ellipse(canvas, (256, 375), (75, 20), 0, 0, 360, col, t, cv2.LINE_AA)
    elif category == "Bed":
        cv2.rectangle(canvas, (110, 110), (402, 260), col, t)
        for x in range(145, 400, 35):
            cv2.line(canvas, (x, 110), (x, 260), col, 2, cv2.LINE_AA)
        cv2.rectangle(canvas, (130, 225), (240, 275), col, 2)
        cv2.rectangle(canvas, (272, 225), (382, 275), col, 2)
        pts_m = np.array([[110, 260], [402, 260], [430, 395], [82, 395]], np.int32)
        cv2.polylines(canvas, [pts_m], True, col, t, cv2.LINE_AA)
        cv2.line(canvas, (92, 335), (420, 335), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (82, 395), (82, 425), col, 5, cv2.LINE_AA)
        cv2.line(canvas, (430, 395), (430, 425), col, 5, cv2.LINE_AA)
        cv2.line(canvas, (82, 425), (430, 425), col, 3, cv2.LINE_AA)
    elif category == "Cabinet":
        cv2.rectangle(canvas, (95, 170), (417, 355), col, t)
        cv2.line(canvas, (85, 160), (427, 160), col, t, cv2.LINE_AA)
        cv2.line(canvas, (85, 160), (95, 170), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (427, 160), (417, 170), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (202, 170), (202, 355), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (310, 170), (310, 355), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (202, 230), (310, 230), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (202, 290), (310, 290), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (185, 245), (185, 275), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (327, 245), (327, 275), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (120, 355), (120, 420), col, 4, cv2.LINE_AA)
        cv2.line(canvas, (392, 355), (392, 420), col, 4, cv2.LINE_AA)
        cv2.line(canvas, (110, 420), (402, 420), col, 3, cv2.LINE_AA)
    else:  # Chair
        pts_back = np.array([[180, 120], [256, 100], [332, 120], [340, 240], [320, 270], [192, 270], [172, 240]], np.int32)
        cv2.polylines(canvas, [pts_back], True, col, t, cv2.LINE_AA)
        cv2.ellipse(canvas, (256, 175), (55, 60), 0, 0, 360, col, 2, cv2.LINE_AA)
        pts_seat = np.array([[150, 260], [362, 260], [370, 320], [340, 335], [172, 335], [142, 320]], np.int32)
        cv2.polylines(canvas, [pts_seat], True, col, t, cv2.LINE_AA)
        cv2.line(canvas, (160, 295), (352, 295), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (172, 220), (145, 270), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (145, 270), (155, 310), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (340, 220), (367, 270), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (367, 270), (357, 310), col, 2, cv2.LINE_AA)
        cv2.line(canvas, (180, 335), (150, 440), col, 4, cv2.LINE_AA)
        cv2.line(canvas, (332, 335), (362, 440), col, 4, cv2.LINE_AA)
        cv2.line(canvas, (205, 330), (185, 415), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (307, 330), (327, 415), col, 3, cv2.LINE_AA)
        cv2.line(canvas, (168, 385), (344, 385), col, 2, cv2.LINE_AA)

    return canvas


## 3. Load or Upload a Sketch & Inspect OpenCV Edges

In [ ]:
#@title Select Sample Preset or Upload Custom Sketch
SKETCH_SOURCE = "Sample Preset"  #@param ["Sample Preset", "Upload Custom Image"]
PRESET_CATEGORY = "Chair"  #@param ["Chair", "Sofa", "Table", "Bed", "Cabinet"]
CUSTOM_IMAGE_PATH = ""  #@param {type:"string"}
EDGE_METHOD = "canny"  #@param ["canny", "adaptive_threshold", "lineart"]
CANNY_LOW = 100  #@param {type:"slider", min:20, max:200, step:5}
CANNY_HIGH = 200  #@param {type:"slider", min:80, max:300, step:5}

raw_bgr = None
if SKETCH_SOURCE == "Upload Custom Image":
    if CUSTOM_IMAGE_PATH and os.path.exists(CUSTOM_IMAGE_PATH):
        raw_bgr = cv2.imread(CUSTOM_IMAGE_PATH, cv2.IMREAD_UNCHANGED)
        print(f"Loaded sketch from path: {CUSTOM_IMAGE_PATH}")
    elif IN_COLAB:
        from google.colab import files
        print("Please upload a furniture sketch (.png, .jpg, .jpeg)...")
        uploaded = files.upload()
        if uploaded:
            fname = next(iter(uploaded.keys()))
            raw_bgr = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_UNCHANGED)
            print(f"Uploaded sketch: {fname}")

if raw_bgr is None:
    sample_file = Path(f"backend/sample_sketches/{PRESET_CATEGORY.lower()}_sketch.png")
    if sample_file.exists():
        raw_bgr = cv2.imread(str(sample_file), cv2.IMREAD_UNCHANGED)
        print(f"Loaded workspace sample sketch: {sample_file}")
    else:
        raw_bgr = generate_sample_sketch(PRESET_CATEGORY)
        print(f"Generated clean {PRESET_CATEGORY} blueprint sketch (512x512).")

prep_result = OpenCVPreprocessor.process_sketch(
    raw_bgr,
    target_size=512,
    low_threshold=CANNY_LOW,
    high_threshold=CANNY_HIGH,
    method=EDGE_METHOD,
)

print(
    f"Background Polarity: {prep_result['detected_bg']} | "
    f"Edge Density: {prep_result['edge_density_percent']}% | "
    f"Contours: {prep_result['contour_count']}"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
axes[0].imshow(cv2.cvtColor(prep_result["padded_bgr"], cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Padded Sketch (512x512)", fontweight="bold")
axes[1].imshow(prep_result["edge_map"], cmap="gray")
axes[1].set_title(f"2. ControlNet Edge Map ({EDGE_METHOD})", fontweight="bold")
axes[2].imshow(prep_result["display_sketch"], cmap="gray")
axes[2].set_title("3. Morphologically Cleaned Lineart", fontweight="bold")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Technique 2 — Pretrained ResNet-50 CNN & Grad-CAM Pixel-by-Pixel Analysis
Runs an ImageNet-pretrained **ResNet-50** convolutional neural network over the preprocessed sketch:
- Registers forward & backward hooks on `layer4` (final spatial feature maps).
- Computes zero-shot furniture probability priors across `{Chair, Sofa, Table, Bed, Cabinet}`.
- Back-propagates gradients to generate a dense **512×512 Grad-CAM pixel attention heatmap** (262,144 pixels scored) plus geometric symmetry & aspect ratio metrics.

In [ ]:
import torchvision.models as models

FURNITURE_CLASS_GROUPS = {
    "Chair": ["barber chair", "folding chair", "rocking chair", "throne"],
    "Sofa": ["studio couch"],
    "Table": ["dining table", "desk"],
    "Bed": ["four-poster", "crib", "cradle"],
    "Cabinet": ["china cabinet", "wardrobe", "chiffonier", "file",
                "medicine chest", "entertainment center", "bookcase"],
}


class CNNFeatureExtractor:
    def __init__(self):
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        self.model = models.resnet50(weights=weights)
        self.model.eval()
        self.preprocess = weights.transforms()
        self.class_names = weights.meta["categories"]

        name_to_idx = {n: i for i, n in enumerate(self.class_names)}
        self.group_indices = {
            g: [name_to_idx[n] for n in names if n in name_to_idx]
            for g, names in FURNITURE_CLASS_GROUPS.items()
        }
        self._activations = None
        self._gradients = None
        self.model.layer4.register_forward_hook(self._save_activation)

    def _save_activation(self, _module, _inp, out):
        self._activations = out
        if out.requires_grad:
            out.register_hook(self._save_gradient)

    def _save_gradient(self, grad):
        self._gradients = grad

    def analyze(self, sketch_bgr: np.ndarray, edge_map: np.ndarray):
        rgb = cv2.cvtColor(sketch_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.preprocess(Image.fromarray(rgb)).unsqueeze(0)

        self.model.zero_grad(set_to_none=True)
        tensor.requires_grad_(True)
        logits = self.model(tensor)
        probs = torch.softmax(logits, dim=1)[0]

        probs_det = probs.detach()
        group_probs = {g: float(probs_det[idx].sum()) for g, idx in self.group_indices.items()}
        mass = sum(group_probs.values())
        predicted = max(group_probs, key=group_probs.get)

        score = logits[0, self.group_indices[predicted]].sum()
        score.backward()
        acts = self._activations.detach()[0]
        grads = self._gradients.detach()[0]
        weights = grads.mean(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * acts).sum(0)).cpu().numpy()
        embedding = acts.mean(dim=(1, 2)).cpu().numpy()

        # Render 512x512 Grad-CAM overlay
        h, w = sketch_bgr.shape[:2]
        cam_norm = cam - cam.min()
        if cam_norm.max() > 0:
            cam_norm = cam_norm / cam_norm.max()
        cam_full = np.clip(cv2.resize(cam_norm.astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC), 0.0, 1.0)
        heat_bgr = cv2.applyColorMap((cam_full * 255).astype(np.uint8), cv2.COLORMAP_JET)
        overlay_bgr = cv2.addWeighted(sketch_bgr, 0.45, heat_bgr, 0.55, 0)
        peak_y, peak_x = np.unravel_index(int(np.argmax(cam_full)), cam_full.shape)

        # Geometric symmetry & aspect ratio
        thick = cv2.dilate(edge_map, np.ones((7, 7), np.uint8))
        left = thick[:, : w // 2]
        right = cv2.flip(thick[:, w - w // 2 :], 1)
        overlap = np.count_nonzero(cv2.bitwise_and(left, right))
        symmetry = 2 * overlap / max(1, np.count_nonzero(left) + np.count_nonzero(right))

        contours, _ = cv2.findContours(edge_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            _, _, bw, bh = cv2.boundingRect(np.vstack(contours))
            aspect = bw / max(1, bh)
        else:
            aspect = 1.0

        norm = mass if mass > 0 else 1.0
        category_scores = {g: round(p / norm, 4) for g, p in group_probs.items()}
        top5 = torch.topk(probs_det, 5)

        return {
            "detected_category": predicted,
            "category_confidence": category_scores[predicted],
            "all_category_scores": category_scores,
            "imagenet_top5": [(self.class_names[i], round(float(p), 4)) for p, i in zip(top5.values.tolist(), top5.indices.tolist())],
            "embedding_dim": int(embedding.shape[0]),
            "embedding_l2_norm": round(float(np.linalg.norm(embedding)), 3),
            "symmetry_score": round(float(symmetry), 3),
            "aspect_ratio": round(float(aspect), 2),
            "contour_count": len(contours),
            "pixels_scored": int(w * h),
            "high_attention_percent": round(float((cam_full > 0.5).mean() * 100), 2),
            "peak_xy": (int(peak_x), int(peak_y)),
            "overlay_rgb": cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB),
            "cam_full": cam_full,
        }


print("Loading ResNet-50 (ImageNet-pretrained)...")
cnn_extractor = CNNFeatureExtractor()
cnn_analysis = cnn_extractor.analyze(prep_result["padded_bgr"], prep_result["edge_map"])

print(
    f"Detected Category: {cnn_analysis['detected_category']} "
    f"(Confidence: {cnn_analysis['category_confidence']*100:.1f}%) | "
    f"Symmetry: {cnn_analysis['symmetry_score']} | Aspect Ratio: {cnn_analysis['aspect_ratio']}"
)
print(f"Grad-CAM Scored {cnn_analysis['pixels_scored']:,} pixels | Peak Attention: {cnn_analysis['peak_xy']}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].imshow(cv2.cvtColor(prep_result["padded_bgr"], cv2.COLOR_BGR2RGB))
axes[0].set_title("Input Sketch (512x512)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(cnn_analysis["overlay_rgb"])
px, py = cnn_analysis["peak_xy"]
axes[1].scatter([px], [py], c="white", s=120, marker="+", linewidths=2.5, label=f"Peak ({px}, {py})")
axes[1].set_title("ResNet-50 Grad-CAM Pixel Attention", fontweight="bold")
axes[1].legend(loc="lower right", framealpha=0.8)
axes[1].axis("off")

cats = list(cnn_analysis["all_category_scores"].keys())
scores = [cnn_analysis["all_category_scores"][c] * 100 for c in cats]
colors = ["#8B5A2B" if c == cnn_analysis["detected_category"] else "#D6CFC2" for c in cats]
bars = axes[2].barh(cats, scores, color=colors)
axes[2].set_xlim(0, 105)
axes[2].set_xlabel("Normalized Category Prior (%)")
axes[2].set_title("CNN Shape Category Scores", fontweight="bold")
for bar, val in zip(bars, scores):
    axes[2].text(val + 1.5, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## 5. Techniques 3, 4 & 5 — CLIP Prompt Synthesizer + Stable Diffusion 1.5 + ControlNet + LCM-LoRA
- **Stable Diffusion 1.5 (`stable-diffusion-v1-5/stable-diffusion-v1-5`)**: Latent diffusion U-Net & VAE.
- **ControlNet Canny (`lllyasviel/control_v11p_sd15_canny`)**: Injects OpenCV structural edge maps directly into U-Net skip connections.
- **LCM-LoRA (`latent-consistency/lcm-lora-sdv1-5`)**: Optional 4–8 step consistency distillation for fast inference.
- **Memory Optimization**: Enables `enable_attention_slicing(1)`, `enable_vae_slicing()`, and `torch.inference_mode()`.

In [ ]:
from diffusers import (
    ControlNetModel,
    LCMScheduler,
    StableDiffusionControlNetImg2ImgPipeline,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

STYLE_DESC = {
    "Modern": "modern contemporary design, clean lines",
    "Minimalist": "minimalist japandi design, simple refined silhouette",
    "Traditional": "traditional classic craftsmanship, detailed woodwork",
    "Luxury": "luxury high-end designer piece, premium details",
    "Scandinavian": "scandinavian nordic design, light and functional",
    "Industrial": "industrial loft design, raw honest materials",
}

MATERIAL_DESC = {
    "Wood": "solid wood with natural wood grain",
    "Metal": "metal frame",
    "Glass": "glass surfaces, transparent tempered glass",
    "Leather": "leather upholstery, stitched seams",
    "Fabric": "fabric upholstery, woven textile texture",
}

FINISH_DESC = {
    "Matte": "matte finish",
    "Glossy": "glossy lacquered finish, reflections",
    "Polished": "polished finish, soft sheen",
    "Brushed": "brushed finish, fine linear texture",
}

NEGATIVE_PROMPT = (
    "sketch, drawing, line art, pencil, cartoon, illustration, painting, lowres, blurry, "
    "deformed, distorted, extra legs, broken legs, bad proportions, cropped, text, "
    "watermark, logo, signature, people, clutter"
)


def build_prompt(category: str, material: str, color: str, style: str, finish: str, custom_prompt: str = ""):
    """Synthesize CLIP-concise (<77 tokens) furniture prompts placing material/category first."""
    parts = [
        f"a {color.lower()} {material.lower()} {category.lower()}",
        MATERIAL_DESC.get(material, material.lower()),
        FINISH_DESC.get(finish, finish.lower()),
        STYLE_DESC.get(style, style.lower()),
    ]
    if custom_prompt and custom_prompt.strip():
        parts.append(custom_prompt.strip())
    parts.append(
        "professional product photograph, studio lighting, neutral background, "
        "photorealistic, highly detailed, sharp focus"
    )
    return {"positive_prompt": ", ".join(parts), "negative_prompt": NEGATIVE_PROMPT}


class ColabDiffusionEngine:
    SD_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    CONTROLNET_ID = "lllyasviel/control_v11p_sd15_canny"
    LCM_LORA = "latent-consistency/lcm-lora-sdv1-5"

    def __init__(self, device: str, dtype):
        self.device = device
        self.dtype = dtype
        self.lcm_available = False
        self.pipe = None
        self.img2img = None

    @staticmethod
    def _from_pretrained(cls, repo_id: str, **kwargs):
        try:
            return cls.from_pretrained(repo_id, variant="fp16", use_safetensors=True, **kwargs)
        except Exception:
            return cls.from_pretrained(repo_id, **kwargs)

    def load(self):
        import time
        t0 = time.time()
        print(f"Loading ControlNet: {self.CONTROLNET_ID}...")
        controlnet = self._from_pretrained(ControlNetModel, self.CONTROLNET_ID, torch_dtype=self.dtype)

        print(f"Loading Stable Diffusion 1.5: {self.SD_MODEL}...")
        pipe = self._from_pretrained(
            StableDiffusionControlNetPipeline,
            self.SD_MODEL,
            controlnet=controlnet,
            torch_dtype=self.dtype,
            safety_checker=None,
            requires_safety_checker=False,
        )
        pipe.set_progress_bar_config(disable=False)

        self.quality_scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        self.lcm_scheduler = LCMScheduler.from_config(pipe.scheduler.config)

        try:
            print(f"Loading LCM-LoRA distillation adapter: {self.LCM_LORA}...")
            pipe.load_lora_weights(self.LCM_LORA, adapter_name="lcm")
            self.lcm_available = True
        except Exception as exc:
            print(f"LCM-LoRA unavailable (using quality scheduler only): {exc}")

        pipe = pipe.to(self.device)
        try:
            img2img = StableDiffusionControlNetImg2ImgPipeline.from_pipe(pipe)
        except Exception:
            img2img = StableDiffusionControlNetImg2ImgPipeline(**pipe.components)
        img2img.set_progress_bar_config(disable=False)

        for p in (pipe, img2img):
            try:
                p.enable_attention_slicing(1)
            except Exception:
                pass
            if hasattr(p, "enable_vae_slicing"):
                p.enable_vae_slicing()

        self.pipe, self.img2img = pipe, img2img
        print(f"All generative weights loaded on {self.device.upper()} in {time.time() - t0:.1f}s!")

    def generate(
        self,
        control_image: Image.Image,
        init_image: Image.Image,
        prompt: str,
        negative_prompt: str,
        mode: str = "controlnet",
        use_lcm: bool = True,
        steps: int = 4,
        guidance_scale: float = 1.5,
        controlnet_scale: float = 1.0,
        strength: float = 0.85,
        seed: int = 42,
    ) -> Image.Image:
        use_lcm = use_lcm and self.lcm_available
        if use_lcm:
            steps = max(4, min(int(steps), 10))
            guidance_scale = max(1.0, min(float(guidance_scale), 2.0))
        else:
            steps = max(12, min(int(steps), 50))

        scheduler = self.lcm_scheduler if use_lcm else self.quality_scheduler
        self.pipe.scheduler = scheduler
        self.img2img.scheduler = scheduler

        if self.lcm_available:
            if use_lcm:
                self.pipe.enable_lora()
            else:
                self.pipe.disable_lora()

        active_pipe = self.img2img if mode == "img2img" else self.pipe
        generator = torch.Generator(device="cpu").manual_seed(int(seed))

        common = dict(
            prompt=prompt,
            negative_prompt=negative_prompt if guidance_scale > 1.0 else None,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            controlnet_conditioning_scale=float(controlnet_scale),
            control_guidance_end=0.85,
            generator=generator,
        )

        with torch.inference_mode():
            if mode == "img2img":
                result = active_pipe(image=init_image, control_image=control_image, strength=float(strength), **common)
            else:
                result = active_pipe(image=control_image, **common)
            return result.images[0]


engine = ColabDiffusionEngine(device=DEVICE, dtype=DTYPE)
engine.load()


## 6. Customize Material, Color & Style → Generate Photoreal Furniture
Select furniture design parameters below and run the cell to execute ControlNet-locked diffusion rendering directly from your sketch.

In [ ]:
#@title Configure Furniture Render & Generate
import time

FURNITURE_CATEGORY = "Chair"  #@param ["Chair", "Sofa", "Table", "Bed", "Cabinet"]
MATERIAL = "Leather"  #@param ["Wood", "Metal", "Glass", "Leather", "Fabric"]
COLOR = "Cognac Brown"  #@param ["Warm Walnut", "Natural Oak", "Charcoal", "Cognac Brown", "Cream Ivory", "Sage Olive", "Midnight Navy", "Terracotta", "Brushed Brass", "Matte Black", "Emerald Velvet", "Smoked Grey"]
STYLE = "Modern"  #@param ["Modern", "Minimalist", "Traditional", "Luxury", "Scandinavian", "Industrial"]
FINISH = "Polished"  #@param ["Matte", "Glossy", "Polished", "Brushed"]
CUSTOM_PROMPT = ""  #@param {type:"string"}

TRANSLATION_MODE = "controlnet"  #@param ["controlnet", "img2img"]
QUALITY_PRESET = "Auto (GPU=25 steps, CPU=6-step LCM)"  #@param ["Auto (GPU=25 steps, CPU=6-step LCM)", "Quality Studio (25 steps UniPC)", "Fast LCM-LoRA (6 steps)"]
CONTROLNET_SCALE = 0.8  #@param {type:"slider", min:0.3, max:1.5, step:0.05}
IMG2IMG_STRENGTH = 0.85  #@param {type:"slider", min:0.3, max:1.0, step:0.05}
NUM_VARIATIONS = 1  #@param {type:"slider", min:1, max:4, step:1}
SEED = 42  #@param {type:"integer"}

if QUALITY_PRESET == "Quality Studio (25 steps UniPC)":
    use_lcm, steps, cfg = False, 25, 7.5
elif QUALITY_PRESET.startswith("Fast LCM"):
    use_lcm, steps, cfg = True, 6, 1.5
else:
    if DEVICE in ("cuda", "xpu"):
        use_lcm, steps, cfg = False, 25, 7.5
    else:
        use_lcm, steps, cfg = True, 6, 1.5

prompts = build_prompt(FURNITURE_CATEGORY, MATERIAL, COLOR, STYLE, FINISH, CUSTOM_PROMPT)
print("Positive Prompt:", prompts["positive_prompt"])
print(f"Mode: {TRANSLATION_MODE} | Scheduler: {'LCM-LoRA' if use_lcm else 'UniPC'} ({steps} steps, CFG={cfg}) | Variations: {NUM_VARIATIONS}")

control_img = Image.fromarray(cv2.cvtColor(prep_result["edge_map"], cv2.COLOR_GRAY2RGB))
init_img = Image.fromarray(cv2.cvtColor(prep_result["padded_bgr"], cv2.COLOR_BGR2RGB))

generated_renders = []
saved_paths = []
t0 = time.time()

for idx in range(NUM_VARIATIONS):
    cur_seed = int(SEED) + idx
    print(f"Generating variation {idx + 1}/{NUM_VARIATIONS} (seed={cur_seed})...")
    out_img = engine.generate(
        control_image=control_img,
        init_image=init_img,
        prompt=prompts["positive_prompt"],
        negative_prompt=prompts["negative_prompt"],
        mode=TRANSLATION_MODE,
        use_lcm=use_lcm,
        steps=steps,
        guidance_scale=cfg,
        controlnet_scale=CONTROLNET_SCALE,
        strength=IMG2IMG_STRENGTH,
        seed=cur_seed,
    )
    out_path = OUTPUT_DIR / f"furnforge_{FURNITURE_CATEGORY.lower()}_{MATERIAL.lower()}_seed{cur_seed}.png"
    out_img.save(out_path)
    generated_renders.append((cur_seed, out_img))
    saved_paths.append(out_path)

elapsed = time.time() - t0
print(f"Completed {NUM_VARIATIONS} render(s) in {elapsed:.1f}s. Saved to: {[str(p) for p in saved_paths]}")

# Plot End-to-End Visual Comparison
n_cols = 3 + len(generated_renders)
fig, axes = plt.subplots(1, n_cols, figsize=(4.5 * n_cols, 4.8))

axes[0].imshow(cv2.cvtColor(prep_result["padded_bgr"], cv2.COLOR_BGR2RGB))
axes[0].set_title("Input Sketch", fontweight="bold")

axes[1].imshow(prep_result["edge_map"], cmap="gray")
axes[1].set_title("ControlNet Canny Edges", fontweight="bold")

axes[2].imshow(cnn_analysis["overlay_rgb"])
axes[2].set_title("ResNet-50 Grad-CAM Scan", fontweight="bold")

for idx, (cur_seed, render_img) in enumerate(generated_renders):
    axes[3 + idx].imshow(render_img)
    axes[3 + idx].set_title(f"Photoreal Render (Seed {cur_seed})", fontweight="bold")

for ax in axes:
    ax.axis("off")

plt.suptitle(f"FurnForge AI: {STYLE} {COLOR} {MATERIAL} {FURNITURE_CATEGORY} ({FINISH})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Save Side-by-Side Showcase & Download Output

In [ ]:
#@title Save Side-by-Side Comparison Sheet & Download (Colab)
AUTO_DOWNLOAD_IN_COLAB = True  #@param {type:"boolean"}

if generated_renders:
    sketch_pil = Image.fromarray(cv2.cvtColor(prep_result["padded_bgr"], cv2.COLOR_BGR2RGB))
    primary_seed, primary_render = generated_renders[0]

    sheet = Image.new("RGB", (1024, 512), (255, 255, 255))
    sheet.paste(sketch_pil.resize((512, 512)), (0, 0))
    sheet.paste(primary_render.resize((512, 512)), (512, 0))

    comparison_path = OUTPUT_DIR / f"comparison_{FURNITURE_CATEGORY.lower()}_seed{primary_seed}.png"
    sheet.save(comparison_path)
    print(f"Saved Before/After showcase sheet: {comparison_path}")
    print(f"Saved Primary Photoreal Render:   {saved_paths[0]}")

    if IN_COLAB and AUTO_DOWNLOAD_IN_COLAB:
        from google.colab import files
        files.download(str(saved_paths[0]))
        files.download(str(comparison_path))


## 8. Optional Interactive Web Studio in Colab (Gradio)
Set `LAUNCH_INTERACTIVE_STUDIO = True` and run this cell in Google Colab to open a live, shareable interactive UI with drag-and-drop sketch upload, OpenCV Canny inspection, ResNet-50 Grad-CAM overlay, and photorealistic generation.

In [ ]:
#@title Launch Interactive Gradio Web Studio (Optional)
LAUNCH_INTERACTIVE_STUDIO = False  #@param {type:"boolean"}

if LAUNCH_INTERACTIVE_STUDIO:
    try:
        import gradio as gr
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
        import gradio as gr

    def gradio_forge(sketch_rgb, category, material, color, style, finish, custom_prompt, mode, quality_preset, cn_scale, strength, seed):
        if sketch_rgb is None:
            bgr = generate_sample_sketch(category)
        elif sketch_rgb.ndim == 2:
            bgr = cv2.cvtColor(sketch_rgb, cv2.COLOR_GRAY2BGR)
        elif sketch_rgb.shape[2] == 4:
            bgr = OpenCVPreprocessor.ensure_bgr(cv2.cvtColor(sketch_rgb, cv2.COLOR_RGBA2BGRA))
        else:
            bgr = cv2.cvtColor(sketch_rgb[:, :, :3], cv2.COLOR_RGB2BGR)

        prep = OpenCVPreprocessor.process_sketch(bgr)
        cnn_out = cnn_extractor.analyze(prep["padded_bgr"], prep["edge_map"])
        prompts = build_prompt(category, material, color, style, finish, custom_prompt)

        use_lcm = quality_preset.startswith("Fast") or (quality_preset.startswith("Auto") and DEVICE not in ("cuda", "xpu"))
        steps = 6 if use_lcm else 25
        cfg = 1.5 if use_lcm else 7.5

        control_img = Image.fromarray(cv2.cvtColor(prep["edge_map"], cv2.COLOR_GRAY2RGB))
        init_img = Image.fromarray(cv2.cvtColor(prep["padded_bgr"], cv2.COLOR_BGR2RGB))

        render = engine.generate(
            control_image=control_img,
            init_image=init_img,
            prompt=prompts["positive_prompt"],
            negative_prompt=prompts["negative_prompt"],
            mode=mode,
            use_lcm=use_lcm,
            steps=steps,
            guidance_scale=cfg,
            controlnet_scale=cn_scale,
            strength=strength,
            seed=int(seed),
        )
        summary = (
            f"CNN Detected: {cnn_out['detected_category']} ({cnn_out['category_confidence']*100:.1f}%) | "
            f"Symmetry: {cnn_out['symmetry_score']} | Edges: {prep['edge_density_percent']}%"
        )
        return prep["edge_map"], cnn_out["overlay_rgb"], render, summary

    with gr.Blocks(title="FurnForge AI — Colab Studio") as demo:
        gr.Markdown("# FurnForge AI — Interactive Sketch-to-Photoreal Furniture Studio")
        with gr.Row():
            with gr.Column():
                in_sketch = gr.Image(label="Upload Furniture Sketch (or leave blank for preset)", type="numpy")
                in_cat = gr.Dropdown(["Chair", "Sofa", "Table", "Bed", "Cabinet"], value="Chair", label="Category")
                in_mat = gr.Dropdown(["Wood", "Metal", "Glass", "Leather", "Fabric"], value="Leather", label="Material")
                in_col = gr.Dropdown(["Warm Walnut", "Natural Oak", "Charcoal", "Cognac Brown", "Cream Ivory", "Sage Olive", "Midnight Navy", "Terracotta", "Brushed Brass", "Matte Black"], value="Cognac Brown", label="Color")
                in_style = gr.Dropdown(["Modern", "Minimalist", "Traditional", "Luxury", "Scandinavian", "Industrial"], value="Modern", label="Style")
                in_fin = gr.Dropdown(["Matte", "Glossy", "Polished", "Brushed"], value="Polished", label="Finish")
                in_custom = gr.Textbox(label="Additional Details (Optional)", placeholder="e.g., stitched seams, walnut legs")
                in_mode = gr.Radio(["controlnet", "img2img"], value="controlnet", label="Translation Mode")
                in_qual = gr.Radio(["Auto", "Quality (25 steps)", "Fast LCM (6 steps)"], value="Auto", label="Sampling Mode")
                in_cn = gr.Slider(0.3, 1.5, value=0.8, step=0.05, label="ControlNet Conditioning Scale")
                in_str = gr.Slider(0.3, 1.0, value=0.85, step=0.05, label="Img2Img Strength")
                in_seed = gr.Number(value=42, precision=0, label="Seed")
                btn = gr.Button("Forge Photoreal Furniture", variant="primary")
            with gr.Column():
                out_render = gr.Image(label="Generated Photoreal Render")
                out_metrics = gr.Textbox(label="CNN & OpenCV Diagnostics")
                with gr.Row():
                    out_edge = gr.Image(label="OpenCV Canny Edges")
                    out_cam = gr.Image(label="ResNet-50 Grad-CAM Scan")

        btn.click(
            gradio_forge,
            inputs=[in_sketch, in_cat, in_mat, in_col, in_style, in_fin, in_custom, in_mode, in_qual, in_cn, in_str, in_seed],
            outputs=[out_edge, out_cam, out_render, out_metrics],
        )

    demo.launch(share=IN_COLAB, debug=False)
else:
    print("Gradio UI skipped (set LAUNCH_INTERACTIVE_STUDIO = True to launch web UI).")
